In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import os
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "urban_model_v2.pth"
IMG_DIR = "../data/finaltestimages"

CATEGORIES = ["Słupy Elektryczne", "Uszkodzona droga", "Uszkodzony Znak","Powalone drzewa", "Smieci", "Graffiti"]

test_transforms = transforms.Compose([
    transforms.Resize((624, 624)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def load_model(path, num_classes):
    model = models.resnet18(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model

def run_test():
    if not os.path.exists(IMG_DIR):
        print(f"Błąd: Katalog {IMG_DIR} nie istnieje!")
        return

    model = load_model(MODEL_PATH, len(CATEGORIES))
    images = [f for f in os.listdir(IMG_DIR) if f.lower().endswith(('png', 'jpg', 'jpeg'))]

    print(f"Znaleziono {len(images)} zdjęć do testu.\n")
    print(f"{'Plik':<25} | {'Przewidywana klasa':<20} | {'Pewność'}")
    print("-" * 65)

    with torch.no_grad():
        for img_name in images:
            img_path = os.path.join(IMG_DIR, img_name)
            
            img = Image.open(img_path).convert('RGB')
            img_tensor = test_transforms(img).unsqueeze(0).to(DEVICE)

            outputs = model(img_tensor)
            
            probabilities = F.softmax(outputs, dim=1)[0]
            prob_percent, class_idx = torch.max(probabilities, 0)

            category = CATEGORIES[class_idx]
            print(f"{img_name:<25} | {category:<20} | {prob_percent.item()*100:>6.2f}%")
            
            prob_list = [f"{p.item()*100:.1f}%" for p in probabilities]
            print(f"   ∟ Rozkład: {dict(zip(CATEGORIES, prob_list))}\n")

if __name__ == "__main__":
    run_test()

C:\Users\mkuzm\AppData\Local\Temp\ipykernel_9556\988319374.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path, map_location=DEVICE))


Znaleziono 1 zdjęć do testu.

Plik                      | Przewidywana klasa   | Pewność
-----------------------------------------------------------------
20260408_201438.jpg       | Smieci               |  76.29%
   ∟ Rozkład: {'Słupy Elektryczne': '0.3%', 'Uszkodzona droga': '20.0%', 'Uszkodzony Znak': '0.0%', 'Powalone drzewa': '1.8%', 'Smieci': '76.3%', 'Graffiti': '1.6%'}

